# MLOps Platform Deployment

This notebook deploys the entire DevOps/Platform stack for the Immich MLOps project, step by step. Each cell runs one discrete action. Run cells top to bottom.

**Before starting, make sure you have:**
1. Your SSH public key added on KVM@TACC (Identity → Key Pairs)
2. An Application Credential downloaded as `clouds.yaml` (Identity → Application Credentials)
3. Your project ID (`CHI-251409`) and netID (`dc6008`)

**This is a Bash notebook.** Make sure the kernel in the top-right says `Bash`. If it says `Python 3`, click it and switch to Bash.

**If the kernel restarts or you close & reopen the notebook, start from Stage 0 (the PATH cell) again.** Environment variables don't persist across kernel restarts.

---
## Stage 0: Configure PATH

Run this cell first, and re-run it if your kernel ever restarts.

In [1]:
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local
echo "PATH configured. Current directory: $(pwd)"

PATH configured. Current directory: /work


---
## Stage 1: Install Terraform and Ansible

One-time install. Takes ~2 minutes.

In [2]:
mkdir -p /work/.local/bin
cd /tmp
wget -q https://releases.hashicorp.com/terraform/1.14.4/terraform_1.14.4_linux_amd64.zip
unzip -o -q terraform_1.14.4_linux_amd64.zip
mv -f terraform /work/.local/bin/
rm terraform_1.14.4_linux_amd64.zip
terraform version

Terraform v1.14.4
on linux_amd64

Your version of Terraform is out of date! The latest version
is 1.14.8. You can update by downloading from https://developer.hashicorp.com/terraform/install


In [3]:
PYTHONUSERBASE=/work/.local pip install --user --quiet ansible-core==2.16.9 ansible==9.8.0
ansible --version | head -1

ansible [core 2.16.9]


---
## Stage 2: Clone your team repo (if not already cloned)

Skip this cell if `/work/mlops-project` already exists from when you cloned earlier.

In [ ]:
if [ ! -d /work/mlops-project ]; then
    cd /work
    git clone https://github.com/SkullMag/mlops-project.git
    echo "Cloned."
else
    echo "Already cloned at /work/mlops-project"
    cd /work/mlops-project && git pull
fi
ls /work/mlops-project/mlops/

---
## Stage 3: Add Kubespray

Kubespray is a separate project we use to install Kubernetes. We clone it as-is into `mlops/ansible/k8s/kubespray/`.

In [4]:
KUBESPRAY_DIR=/work/mlops-project/mlops/ansible/k8s/kubespray
if [ ! -d "$KUBESPRAY_DIR" ]; then
    git clone -b release-2.26 https://github.com/kubernetes-sigs/kubespray.git "$KUBESPRAY_DIR"
    echo "Kubespray cloned."
else
    echo "Kubespray already present."
fi
ls "$KUBESPRAY_DIR" | head

Cloning into '/work/mlops-project/mlops/ansible/k8s/kubespray'...
remote: Enumerating objects: 87008, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 87008 (delta 67), reused 19 (delta 19), pack-reused 86903 (from 3)
Receiving objects: 100% (87008/87008), 28.33 MiB | 4.98 MiB/s, done.
Resolving deltas: 100% (48878/48878), done.
Kubespray cloned.
CHANGELOG.md
CNAME
CONTRIBUTING.md
Dockerfile
LICENSE
Makefile
OWNERS
OWNERS_ALIASES
README.md
RELEASE.md


In [5]:
# Install Kubespray's Python dependencies
PYTHONUSERBASE=/work/.local pip install --user --quiet -r /work/mlops-project/mlops/ansible/k8s/kubespray/requirements.txt
echo "Kubespray requirements installed."

Kubespray requirements installed.


---
## Stage 4: Configure credentials

**Before running the cell below:**

1. Open the file browser on the left. Navigate to `/work/mlops-project/mlops/tf/kvm/`.
2. Double-click `clouds.yaml.template` to open it.
3. Replace `REDACTED_UNIQUE_ID` and `REDACTED_SECRET` with values from the `clouds.yaml` you downloaded from KVM@TACC.
4. **File → Save As** → save as `clouds.yaml` (same directory, drop the `.template`).

Then run this cell to verify:

In [6]:
if [ -f /work/mlops-project/mlops/tf/kvm/clouds.yaml ]; then
    echo "✓ clouds.yaml exists"
    # Sanity check: no REDACTED markers
    if grep -q REDACTED /work/mlops-project/mlops/tf/kvm/clouds.yaml; then
        echo "✗ clouds.yaml still has REDACTED placeholders! Edit it first."
    else
        echo "✓ No REDACTED placeholders found"
    fi
else
    echo "✗ clouds.yaml NOT found. Create it by copying clouds.yaml.template and filling in credentials."
fi

✓ clouds.yaml exists
✓ No REDACTED placeholders found


---
## Stage 5: Create server reservation

Uses OpenStack CLI (pre-installed in Jupyter) to reserve 3 `m1.large` VMs for 12 hours.

In [8]:
# Set OpenStack auth (uses your Chameleon login, NOT application credentials)
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-251409"
export OS_REGION_NAME="KVM@TACC"

# Create the reservation
openstack reservation lease create lease_mlops_proj12 \
  --start-date "$(date -u -d '+30 seconds' '+%Y-%m-%d %H:%M')" \
  --end-date "$(date -u -d '+12 hours' '+%Y-%m-%d %H:%M')" \
  --reservation "resource_type=flavor:instance,flavor_id=$(openstack flavor show m1.large -f value -c id),amount=3" \
  2>&1 | tail -20
echo "---"
echo "Wait ~30 seconds for the lease to become ACTIVE, then run the next cell."

|              |         "status": "pending",                                                                                                                                                                                                                                                                                                                           |
|              |         "missing_resources": false,                                                                                                                                                                                                                                                                                                                    |
|              |         "resources_changed": false,                                                                                                                                                                                                                                

In [9]:
# Retrieve the reservation flavor ID (needed by Terraform).
# If the lease isn't ACTIVE yet, this will fail - wait a bit and re-run.
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-251409"
export OS_REGION_NAME="KVM@TACC"

export FLAVOR_ID=$(openstack reservation lease show lease_mlops_proj12 -f json -c reservations \
      | python3 -c 'import sys, json; r = json.load(sys.stdin)["reservations"][0]; r = json.loads(r) if isinstance(r, str) else r; print(r["flavor_id"])')

echo "Reservation flavor ID: $FLAVOR_ID"
echo "export FLAVOR_ID=$FLAVOR_ID" > /work/.mlops_env

Reservation flavor ID: 80707b52-37ec-4d37-987b-922cf882033c


---
## Stage 6: Terraform — provision 3 VMs

Note the **floating IP** printed at the end. You'll use it in several places.

In [10]:
cd /work/mlops-project/mlops/tf/kvm

# Clear OS_* variables so Terraform uses clouds.yaml exclusively
unset $(set | grep -o "^OS_[A-Za-z0-9_]*")

# Load the flavor ID from earlier
source /work/.mlops_env

# Terraform variables
export TF_VAR_suffix=proj12
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$FLAVOR_ID

echo "TF_VAR_suffix=$TF_VAR_suffix"
echo "TF_VAR_key=$TF_VAR_key"
echo "TF_VAR_reservation=$TF_VAR_reservation"

terraform init

TF_VAR_suffix=proj12
TF_VAR_key=id_rsa_chameleon
TF_VAR_reservation=80707b52-37ec-4d37-987b-922cf882033c
Initializing the backend...
Initializing provider plugins...
- Finding terraform-provider-openstack/openstack versions matching "~> 1.51.1"...
- Installing terraform-provider-openstack/openstack v1.51.1...
- Installed terraform-provider-openstack/openstack v1.51.1 (self-signed, key ID 4F80527A391BEFD2)
Partner and community providers are signed by their developers.
If you'd like to know more about provider signing, you can read about it here:
https://developer.hashicorp.com/terraform/cli/plugins/signing
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform pl

In [11]:
cd /work/mlops-project/mlops/tf/kvm
source /work/.mlops_env
export TF_VAR_suffix=proj12
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$FLAVOR_ID

terraform validate && terraform plan | tail -40

Success! The configuration is valid.

          + "80d5182b-854c-465e-b2ce-aacd8302f28d",
          + "c8ac33cf-a542-4640-9887-396fe417f956",
          + "fa9d35f5-5f06-409b-9cf0-431db8021ee0",
        ]
      + tenant_id              = (known after apply)

      + binding (known after apply)
    }

  # openstack_networking_subnet_v2.private_subnet will be created
  + resource "openstack_networking_subnet_v2" "private_subnet" {
      + all_tags          = (known after apply)
      + cidr              = "192.168.1.0/24"
      + enable_dhcp       = true
      + gateway_ip        = (known after apply)
      + id                = (known after apply)
      + ip_version        = 4
      + ipv6_address_mode = (known after apply)
      + ipv6_ra_mode      = (known after apply)
      + name              = "private-subnet-mlops-proj12"
      + network_id        = (known after apply)
      + no_gateway        = true
      + region            = (known after apply)
      + service_types     = (know

In [12]:
cd /work/mlops-project/mlops/tf/kvm
source /work/.mlops_env
export TF_VAR_suffix=proj12
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$FLAVOR_ID

terraform apply -auto-approve 2>&1 | tail -15
echo "---"
FLOATING_IP=$(terraform output -raw floating_ip_out)
echo "✓ Floating IP: $FLOATING_IP"
echo "export FLOATING_IP=$FLOATING_IP" >> /work/.mlops_env

openstack_compute_instance_v2.nodes["node2"]: Creating...
openstack_compute_instance_v2.nodes["node3"]: Creating...
openstack_compute_instance_v2.nodes["node1"]: Creating...
openstack_compute_instance_v2.nodes["node2"]: Still creating... 10s elapsed]
openstack_compute_instance_v2.nodes["node3"]: Still creating... 10s elapsed]
openstack_compute_instance_v2.nodes["node1"]: Still creating... 10s elapsed]
openstack_compute_instance_v2.nodes["node2"]: Creation complete after 11s [id=365a3a75-b248-4fe2-bb4f-94e3ae8fe5da]
openstack_compute_instance_v2.nodes["node1"]: Creation complete after 12s [id=31c73f76-094f-4bc6-8ede-d7c49726bf56]
openstack_compute_instance_v2.nodes["node3"]: Creation complete after 12s [id=bf554813-eb73-4576-9693-e5442e350bca]

Apply complete! Resources: 12 added, 0 changed, 0 destroyed.

Outputs:

floating_ip_out = "129.114.27.118"
---
✓ Floating IP: 129.114.27.118


In [57]:
# Start the agent and add your key
eval $(ssh-agent -s)
ssh-add ~/.ssh/id_rsa_chameleon

# Verify the key is loaded
ssh-add -l

Agent pid 9684
Identity added: /home/dc6008_nyu_edu/.ssh/id_rsa_chameleon (danielacruz@MacBook-Pro-62.local)
3072 SHA256:O2H3xngu3cWV5sckhUvc+W4MfM1Ecwo7fU9WoaDh14o danielacruz@MacBook-Pro-62.local (RSA)


---
## Stage 7: Wire the floating IP into Ansible config

The Ansible SSH-jump proxy needs the floating IP we just got.

In [58]:
source /work/.mlops_env
echo "Substituting floating IP $FLOATING_IP into ansible.cfg..."
sed -i "s/cc@A\.B\.C\.D/cc@$FLOATING_IP/" /work/mlops-project/mlops/ansible/ansible.cfg
grep ProxyCommand /work/mlops-project/mlops/ansible/ansible.cfg

Substituting floating IP 129.114.27.118 into ansible.cfg...
           -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p cc@129.114.27.118"


In [59]:
# Quick connectivity test: Ansible can reach all 3 nodes via SSH jump
cd /work/mlops-project/mlops/ansible
ansible -i inventory.yml all -m ping

node2 | SUCCESS => {
    "changed": false,
    "ping": "pong"
}
node1 | SUCCESS => {
    "changed": false,
    "ping": "pong"
}
node3 | SUCCESS => {
    "changed": false,
    "ping": "pong"
}


---
## Stage 8: Pre-K8s configuration (firewall + Docker registry)

In [60]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml pre_k8s/pre_k8s_configure.yml


PLAY [Disable and Mask firewalld] **********************************************

TASK [Gathering Facts] *********************************************************
ok: [node1]
ok: [node2]
ok: [node3]

TASK [Stop firewalld service] **************************************************
ok: [node1]
ok: [node3]
ok: [node2]

TASK [Mask firewalld service] **************************************************
changed: [node2]
changed: [node1]
changed: [node3]

PLAY [Set up insecure registry for Docker] *************************************

TASK [Gathering Facts] *********************************************************
ok: [node2]
ok: [node1]
ok: [node3]

TASK [Ensure /etc/docker directory exists] *************************************
ok: [node3]
ok: [node1]
ok: [node2]

TASK [Create /etc/docker/daemon.json if not exists] ****************************
changed: [node2]
changed: [node1]
changed: [node3]

TASK [Configure Docker daemon.json for insecure registry] **********************
ok: [node2]
ok: 

---
## Stage 9: Install Kubernetes (Kubespray) — this is the long one

**This takes 30–60 minutes.** Start it, then go do something else for an hour. When you come back, check the output ended with a `PLAY RECAP` showing all nodes succeeded.

In [61]:
cd /work/mlops-project/mlops/ansible/k8s/kubespray
export ANSIBLE_CONFIG=/work/mlops-project/mlops/ansible/ansible.cfg
export ANSIBLE_ROLES_PATH=roles

ansible-playbook -i ../inventory/mycluster --become --become-user=root ./cluster.yml

[WARNING]: While constructing a mapping from /work/mlops-
project/mlops/ansible/k8s/kubespray/roles/bootstrap-os/tasks/main.yml, line 29,
column 7, found a duplicate dict key (paths). Using last defined value only.

PLAY [Check Ansible version] ***************************************************

TASK [Check 2.16.4 <= Ansible version < 2.17.0] ********************************
ok: [node1] => changed=false 
  msg: All assertions passed

TASK [Check that python netaddr is installed] **********************************
ok: [node1] => changed=false 
  msg: All assertions passed

TASK [Check that jinja is not too old (install via pip)] ***********************
ok: [node1] => changed=false 
  msg: All assertions passed
[WARNING]: Could not match supplied host pattern, ignoring: kube-master

PLAY [Add kube-master nodes to kube_control_plane] *****************************
skipping: no hosts matched
[WARNING]: Could not match supplied host pattern, ignoring: kube-node

PLAY [Add kube-node nodes to

---
## Stage 10: Post-K8s — ArgoCD, Argo Workflows, Argo Events

**Critical:** This cell prints two secrets at the end. **Save them.** You'll need them to log into the ArgoCD UI and Kubernetes dashboard later.
- "Dashboard token: ..."
- "ArgoCD admin password: ..."

In [62]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml post_k8s/post_k8s_configure.yml


PLAY [Post-Install kubectl Setup] **********************************************

TASK [Gathering Facts] *********************************************************
ok: [node1]
ok: [node2]

TASK [Ensure .kube directory exists] *******************************************
ok: [node1]
ok: [node2]

TASK [Copy admin.conf to user's kubeconfig] ************************************
ok: [node1]
ok: [node2]

TASK [Run kubectl get nodes as cc] *********************************************
changed: [node1]
changed: [node2]

TASK [Show kubectl get nodes output] *******************************************
ok: [node1] => 
  msg:
  - NAME    STATUS   ROLES           AGE     VERSION
  - node1   Ready    control-plane   3h13m   v1.30.6
  - node2   Ready    control-plane   3h13m   v1.30.6
  - node3   Ready    <none>          3h12m   v1.30.6
ok: [node2] => 
  msg:
  - NAME    STATUS   ROLES           AGE     VERSION
  - node1   Ready    control-plane   3h13m   v1.30.6
  - node2   Ready    control-plane   3

---
## Stage 11: Deploy platform services

This installs MLflow, MinIO, Postgres, Prometheus, Grafana, Alertmanager, kube-state-metrics, and the Traefik gateway via ArgoCD.

**Save the printed Grafana admin password.**

In [63]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_platform.yml


PLAY [Install Gateway API CRDs] ************************************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Apply Gateway API CRDs with kubectl] *************************************
changed: [node1]

PLAY [Deploy MLflow platform via ArgoCD & Helm with secret handling] ***********

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Get ArgoCD admin password from Kubernetes secret] ************************
changed: [node1]

TASK [Decode ArgoCD admin password] ********************************************
changed: [node1]

TASK [Log in to ArgoCD] ********************************************************
ok: [node1]

TASK [Add repository to ArgoCD] ************************************************
changed: [node1]

TASK [Detect external IP starting with 10.56] **********************************
ok: [node1]

TASK [Ensure immich-platform namespace exists] ****************

: 2

---
## Stage 12: Deploy Immich (the open source service)

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_immich.yml

---
## Stage 13: Bootstrap container images

These playbooks run one-shot Argo Workflows that build the initial serving and training container images. Your team must have pushed the training code to the `mlops` branch and the serving code to the `workflow` branch of the team repo for these to succeed. If those branches don't exist yet, these cells will fail — that's OK, you can come back and re-run them later.

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/workflow_build_init.yml

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/workflow_build_training_init.yml

---
## Stage 14: Deploy the three serving environments

In [51]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_staging.yml


PLAY [Deploy Gourmetgram Staging via ArgoCD & Helm] ****************************

TASK [Gathering Facts] *********************************************************
fatal: [node1]: UNREACHABLE! => changed=false 
  msg: |-
    Data could not be sent to remote host "192.168.1.11". Make sure this host can be reached over ssh: Connection closed by UNKNOWN port 65535
  unreachable: true

PLAY RECAP *********************************************************************
node1                      : ok=0    changed=0    unreachable=1    failed=0    skipped=0    rescued=0    ignored=0   



: 4

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_canary.yml

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_prod.yml

---
## Stage 15: Apply Argo Workflow templates + rollback sensor

In [ ]:
cd /work/mlops-project/mlops/ansible
ansible-playbook -i inventory.yml argocd/workflow_templates_apply.yml

In [ ]:
# Apply the rollback sensor on node1 (Argo Events resources, not templates)
source /work/.mlops_env
ssh -o StrictHostKeyChecking=no -i ~/.ssh/id_rsa_chameleon cc@$FLOATING_IP \
  "kubectl apply -f - < /dev/stdin" < /work/mlops-project/mlops/workflows/rollback-sensor.yaml

---
## Stage 16: Verify the deployment

In [ ]:
source /work/.mlops_env
echo "Floating IP: $FLOATING_IP"
echo ""
echo "Checking platform services (via external IPs on the floating IP):"
echo ""
for port_name in "8000 MLflow" "9001 MinIO" "3000 Grafana" "9090 Prometheus"; do
    port=$(echo $port_name | awk '{print $1}')
    name=$(echo $port_name | awk '{print $2}')
    status=$(curl -s -o /dev/null -w "%{http_code}" --max-time 5 http://$FLOATING_IP:$port/ || echo "FAIL")
    echo "  $name (port $port): HTTP $status"
done
echo ""
echo "Checking K8s via SSH on node1:"
ssh -o StrictHostKeyChecking=no -i ~/.ssh/id_rsa_chameleon cc@$FLOATING_IP "kubectl get nodes && echo && kubectl get pods --all-namespaces | grep -v Running | head -10"

---
## Stage 17: Access the UIs from your laptop

For ArgoCD and Argo Workflows, you need to SSH-tunnel from your **local laptop** (not here in Jupyter). In a terminal on your laptop:

```bash
# ArgoCD UI
ssh -L 8888:127.0.0.1:8888 -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
# Then on node1:
kubectl port-forward svc/argocd-server -n argocd 8888:443
# Browse https://127.0.0.1:8888/ (username: admin, password: saved in stage 10)
```

```bash
# Argo Workflows UI
ssh -L 2746:127.0.0.1:2746 -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
kubectl -n argo port-forward svc/argo-server 2746:2746
# Browse https://127.0.0.1:2746/
```

Grafana, Prometheus, MinIO, and MLflow are on external IPs and reachable directly in your browser — no tunnel needed.

- Grafana:    http://<FLOATING_IP>:3000
- Prometheus: http://<FLOATING_IP>:9090
- MinIO:      http://<FLOATING_IP>:9001
- MLflow:     http://<FLOATING_IP>:8000

---
## Stage 99: Tear down (when done, to save Chameleon resources)

Run only when you're completely done with the cluster.

In [ ]:
cd /work/mlops-project/mlops/tf/kvm
source /work/.mlops_env
export TF_VAR_suffix=dc6008
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$FLAVOR_ID
unset $(set | grep -o "^OS_[A-Za-z0-9_]*")

terraform destroy -auto-approve

In [ ]:
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-251409"
export OS_REGION_NAME="KVM@TACC"
openstack reservation lease delete lease_mlops_dc6008